## <center>Khalifa University — Agentic AI Program</center>
### <center>Program 5 · Day 1: Advanced Agentic Workflows</center>
# Exercise 4: Review, Revise, and Stop — Bounded Revision Cycle

**Module:** Reflective Agent Loops & Self-Correction  
**Companion Script:** `Day 1/Excercises/day1_workbook_code/patient_revision.py`  
**Estimated Practical Time:** 30 minutes  
**Target Model:** Local Ollama (`qwen3.5:4b`, `qwen3:4b-instruct`, `qwen2.5:7b`, or `qwen2.5:3b`)

---

### Aim & Learning Objectives
By the end of this exercise, you will be able to:
1. **Extend a sequential workflow into a reflective loop** where reviewer feedback actively triggers a revision.
2. **Enforce a strict, code-level stopping rule** in Python: exactly **one revision**, followed by a mandatory recheck.
3. **Prevent runaway infinite loops** by avoiding unconstrained `while` loops between LLMs.
4. **Supply ground truth to the reviser** (`note`, `draft`, `issues`) so corrections remain anchored to clinical reality.
5. **Generate structured audit records** saved to `outputs/` tracking initial draft, first review, final draft, and final review.

## Architecture: The Bounded Revision Cycle

In earlier exercises, the reviewer was passive: it produced a verdict (`PASS` or `REVISE`), but the workflow stopped there. In Exercise 4, we convert feedback into active self-correction while strictly bounding the cycle to at most **one revision**:

<p align="center">
  <img src="images/exercise4_revision_flow.jpg" alt="Exercise 4: Bounded Reflection Cycle & Stopping Guard" style="max-width: 100%; border-radius: 8px; box-shadow: 0 4px 12px rgba(0,0,0,0.08);">
</p>

### Stopping Rule Decision Tree & Loop Prevention

| Branch / Step | Evaluated Condition | Action Dispatched | Revisions Allowed | Safety Justification |
|---|---|---|---|---|
| **Clean Fast-Path** | `first_review["decision"] == "PASS"` | Return initial draft immediately | `0` | Eliminates wasted inference latency and token costs on valid drafts. |
| **Correction Branch**| `first_review["decision"] == "REVISE"` | Invoke `revise_report()` once | `1` | Provides one bounded opportunity to fix flagged hallucinations. |
| **Mandatory Recheck**| `final_review["decision"] == "PASS"` | Approve revised draft (`ready_for_human_review`) | `1` (Terminal) | Guarantees that corrected draft undergoes independent re-evaluation. |
| **Safety Escalation**| `final_review["decision"] == "REVISE"` | Mark `needs_instructor` for human triage | `1` (Terminal) | **Hard stop:** prevents infinite adversarial ping-pong between models. |

### Audit Log Schema (`outputs/revision_*.json`)

| Field Name | Type | Description |
|---|---|---|
| `model` | `string` | Local LLM identifier used for all inferences |
| `status` | `string` | Final workflow state: `"ready_for_human_review"` or `"needs_instructor"` |
| `revisions` | `integer` | Count of revisions executed (`0` or `1`) |
| `model_requests` | `integer` | Cumulative API calls (`3` on clean pass; `5` on repair) |
| `first_decision` | `string` | Initial reviewer verdict (`"PASS"` or `"REVISE"`) |
| `first_issues` | `list[string]` | Specific issues caught during initial review |
| `final_decision` | `string` | Recheck reviewer verdict (`"PASS"` or `"REVISE"`) |
| `final_issues` | `list[string]` | Any remaining issues after revision |
| `initial_draft` | `string` | Full text of initial draft (before revision) |
| `final_report` | `string` | Full text of approved or escalated clinical report |
| `elapsed_seconds`| `float` | Wall-clock execution time for end-to-end cycle |

In [1]:
import argparse
import json
import os
import sys
import time
from pathlib import Path
from uuid import uuid4
from ollama import chat

MODEL_NAME = os.environ.get("OLLAMA_MODEL", "qwen3:4b-instruct")
OLLAMA_URL = os.environ.get("OLLAMA_URL", "http://localhost:11434")
CONTEXT_TOKENS = 8192
TEMPERATURE = 0
CALL_COUNT = 0

print("Revision Cycle Configuration:")
print(f"  Target Model: {MODEL_NAME}")
print(f"  Ollama URL:   {OLLAMA_URL}")

Revision Cycle Configuration:
  Target Model: qwen3:4b-instruct
  Ollama URL:   http://localhost:11434


## Preflight Check
Verifies Ollama connectivity and auto-detects the active model.

In [2]:
import importlib.util
import json
import urllib.request
import urllib.error

class PreflightError(RuntimeError):
    pass

def run_preflight():
    # 1. Check Python library
    if importlib.util.find_spec("ollama") is None:
        raise PreflightError("The 'ollama' library is missing. Install with: pip install ollama")
    
    # 2. Check Ollama API reachability
    try:
        req = urllib.request.Request(f"{OLLAMA_URL}/api/tags")
        with urllib.request.urlopen(req, timeout=3) as resp:
            data = json.loads(resp.read().decode())
            installed_models = [m.get("name") for m in data.get("models", [])]
    except Exception as e:
        raise PreflightError(
            f"Cannot connect to Ollama at {OLLAMA_URL}. Ensure Ollama is running (`ollama serve`). Details: {e}"
        )
    
    print(f"Ollama server is active at {OLLAMA_URL}.")
    print(f"Installed models detected ({len(installed_models)}):")
    for m in installed_models:
        print(f"  - {m}")
    
    # Smart Model Resolution:
    # 1. Respect explicit environment variable if provided
    # 2. Check preferred MODEL_NAME (qwen3:4b-instruct)
    # 3. Check recommended workshop models (qwen3.5:4b, qwen2.5:7b, qwen2.5:3b)
    # 4. Fall back to any installed qwen / available model
    global MODEL_NAME
    env_model = os.environ.get("OLLAMA_MODEL")
    if env_model:
        MODEL_NAME = env_model
        print(f"\nUsing environment specified model: '{MODEL_NAME}'")
        return
    
    if any(m.startswith(MODEL_NAME) or MODEL_NAME.startswith(m.split(":")[0]) for m in installed_models):
        for m in installed_models:
            if m.startswith(MODEL_NAME) or MODEL_NAME.startswith(m.split(":")[0]):
                MODEL_NAME = m
                break
        print(f"\nActive model selected: '{MODEL_NAME}' (target match)")
        return
    
    for candidate in ["qwen3.5:4b", "qwen2.5:7b", "qwen2.5:3b"]:
        match = next((m for m in installed_models if m.startswith(candidate) or candidate.startswith(m.split(":")[0])), None)
        if match:
            MODEL_NAME = match
            print(f"\n[NOTICE] '{MODEL_NAME}' selected from installed models (aligns with recommended workshop models).")
            return
            
    if installed_models:
        MODEL_NAME = installed_models[0]
        print(f"\n[NOTICE] Selected first available model: '{MODEL_NAME}'")
    else:
        raise PreflightError("No models found in Ollama. Pull a model with: ollama pull qwen2.5:7b")

try:
    run_preflight()
except PreflightError as err:
    print(f"PREFLIGHT CHECK FAILED: {err}")

Ollama server is active at http://localhost:11434.
Installed models detected (3):
  - qwen3.5:4b
  - qwen3:4b-instruct
  - qwen2.5:3b

Active model selected: 'qwen3:4b-instruct' (target match)


## The Fictional Patient Note (Ground Truth)

The patient note below serves as the source of truth for the revision cycle.

In [3]:
PATIENT_NOTE = """
Patient ID: DEMO-001
Age: 60 years
Diagnosis recorded by the clinician: Colon cancer
Reported symptoms: Abdominal pain and blood in the stool
Colonoscopy finding: A mass in the colon
Biopsy result: Adenocarcinoma
Cancer stage: Not provided
Treatment plan: Not provided
""".strip()

print("Loaded Fictional Patient Note (Source of Truth):")
print("-" * 50)
print(PATIENT_NOTE)
print("-" * 50)

Loaded Fictional Patient Note (Source of Truth):
--------------------------------------------------
Patient ID: DEMO-001
Age: 60 years
Diagnosis recorded by the clinician: Colon cancer
Reported symptoms: Abdominal pain and blood in the stool
Colonoscopy finding: A mass in the colon
Biopsy result: Adenocarcinoma
Cancer stage: Not provided
Treatment plan: Not provided
--------------------------------------------------


## Step 1: Base Helper and Role Functions

We establish `ask_model`, `extract_facts`, `write_report`, and `review_report` matching Exercise 1.

In [4]:
def ask_model(instructions, data, json_output=False):
    global CALL_COUNT
    settings = {
        "model": MODEL_NAME,
        "messages": [
            {"role": "system", "content": instructions},
            {"role": "user", "content": data},
        ],
        "options": {"temperature": TEMPERATURE, "num_ctx": CONTEXT_TOKENS},
        "stream": False,
    }
    if json_output:
        settings["format"] = "json" if json_output is True else json_output
    CALL_COUNT += 1
    response = chat(**settings)
    return response.message.content

def extract_facts(note):
    raw = ask_model(
        "Extract exactly these eight fields as a JSON object: patient_id, age, "
        "recorded_diagnosis, symptoms, colonoscopy, biopsy, stage, and treatment. "
        "Use only the note. Use the exact string 'Not provided' for missing facts. "
        "Do not infer stage, explain diagnosis, or recommend treatment. Return JSON only.",
        note,
        json_output=True,
    )
    return json.loads(raw)

def write_report(facts):
    return ask_model(
        "Write a patient summary under 150 words. Include all "
        "eight supplied fields. State when stage and treatment "
        "are not provided. Add no conclusions or recommendations. "
        "Say 'recorded diagnosis'; do not infer how or when the "
        "diagnosis was made or connect it causally to symptoms/tests.",
        json.dumps(facts, indent=2),
    )

def review_report(note, report):
    answer = ask_model(
        "Compare the report with the original note. Check all "
        "eight fields for omissions, errors, and invented facts. "
        "Reject invented explanations of how or when diagnosis was made. "
        "Return JSON with decision (PASS or REVISE) and issues "
        "(a list of specific correction strings). Use PASS only "
        "if there are no issues, with an empty issues list. "
        "Use REVISE with a nonempty issues list otherwise.",
        f"ORIGINAL NOTE:\n{note}\n\nDRAFT:\n{report}",
        json_output=True,
    )
    return json.loads(answer)

print("Base helper and role functions initialized.")

Base helper and role functions initialized.


## Step 2: Define the Revision Role (`revise_report`)

**Revision Role Responsibility:**
- Receives:
  1. The **original note** (source of truth).
  2. The **flawed draft**.
  3. The **specific issues** returned by the reviewer.
- Returns: Corrected draft report under 150 words.

In [5]:
KNOWN_ERROR = (
    "\nThe patient has stage III cancer. "
    "Chemotherapy is planned."
)

def revise_report(note, report, issues):
    return ask_model(
        "Correct the draft using the original note and feedback. "
        "The note is the source of truth. Include all eight fields. "
        "Do not infer stage or treatment. Return only the corrected "
        "report, under 150 words. Say 'recorded diagnosis'; do not "
        "infer how or when it was made or link it to symptoms/tests.",
        f"NOTE:\n{note}\nDRAFT:\n{report}\n"
        f"CORRECTIONS:\n{json.dumps(issues)}",
    )

print("revise_report function defined.")

revise_report function defined.


## Step 3: Implement the Bounded Cycle (`run_cycle`)

Notice the stopping rule:
```python
# A single conditional branch permits at most one revision.
if first_review["decision"] == "REVISE":
    final_report = revise_report(note, draft, first_review["issues"])
    final_review = review_report(note, final_report)
    revisions = 1
```
There is no `while` loop. The cycle either passes initially or performs exactly one revision and mandatory recheck.

In [6]:
def run_cycle(note, inject_error=False):
    global CALL_COUNT
    CALL_COUNT = 0
    start = time.perf_counter()
    
    # 1. Extraction & Writing
    facts = extract_facts(note)
    draft = write_report(facts)
    if inject_error:
        draft += KNOWN_ERROR
        
    # 2. First Review
    first_review = review_report(note, draft)
    final_report = draft
    final_review = first_review
    revisions = 0
    
    # 3. Bounded Revision Branch (at most 1 revision)
    if first_review["decision"] == "REVISE":
        final_report = revise_report(note, draft, first_review["issues"])
        final_review = review_report(note, final_report)
        revisions = 1
        
    status = (
        "ready_for_human_review"
        if final_review["decision"] == "PASS"
        else "needs_instructor"
    )
    
    return {
        "model": MODEL_NAME,
        "status": status,
        "revisions": revisions,
        "model_requests": CALL_COUNT,
        "first_decision": first_review["decision"],
        "first_issues": first_review["issues"],
        "final_decision": final_review["decision"],
        "final_issues": final_review["issues"],
        "initial_draft": draft,
        "final_report": final_report,
        "elapsed_seconds": round(time.perf_counter() - start, 2),
    }

print("Bounded run_cycle pipeline initialized.")

Bounded run_cycle pipeline initialized.


## Step 4: Test Condition A — Clean Run (Expected PASS)

**Prediction:**
- A clean run should pass on the first review.
- Number of revisions: `0`
- Number of model requests: `3` (extract + write + review)
- Expected status: `"ready_for_human_review"`

In [7]:
print("Running Test Condition A (Clean Draft)...")
clean_record = run_cycle(PATIENT_NOTE, inject_error=False)

print(f"\nStatus:          {clean_record['status']}")
print(f"Revisions:       {clean_record['revisions']}")
print(f"Model Requests:  {clean_record['model_requests']}")
print(f"First Decision:  {clean_record['first_decision']}")
print(f"Final Decision:  {clean_record['final_decision']}")
print(f"Elapsed Time:    {clean_record['elapsed_seconds']}s")

assert clean_record["revisions"] == 0, "Clean run should require 0 revisions!"
assert clean_record["status"] == "ready_for_human_review", "Clean run should pass!"
print("\nSUCCESS: Test Condition A verified.")

Running Test Condition A (Clean Draft)...

Status:          ready_for_human_review
Revisions:       0
Model Requests:  3
First Decision:  PASS
Final Decision:  PASS
Elapsed Time:    28.21s

SUCCESS: Test Condition A verified.


## Step 5: Test Condition B — Injected Error & Revision

**Prediction:**
- Initial review returns `REVISE`.
- The revision agent runs once, removing the hallucinated stage III cancer and chemotherapy.
- Mandatory recheck reviews the revised draft.
- Number of revisions: `1`
- Number of model requests: `5` (extract + write + review + revise + recheck review)
- Final status: `"ready_for_human_review"`

In [8]:
print("Running Test Condition B (Injected Error Draft)...")
error_record = run_cycle(PATIENT_NOTE, inject_error=True)

print(f"\nStatus:          {error_record['status']}")
print(f"Revisions:       {error_record['revisions']}")
print(f"Model Requests:  {error_record['model_requests']}")
print(f"First Decision:  {error_record['first_decision']} (Issues: {len(error_record['first_issues'])})")
print(f"Final Decision:  {error_record['final_decision']}")
print(f"Elapsed Time:    {error_record['elapsed_seconds']}s")

print("\n--- INITIAL FLAWED DRAFT ---")
print(error_record["initial_draft"])

print("\n--- FIRST REVIEW ISSUES ---")
for iss in error_record["first_issues"]:
    print(f"  * {iss}")

print("\n--- REVISED FINAL REPORT ---")
print(error_record["final_report"])

# Save classroom execution record
out_dir = Path("outputs")
out_dir.mkdir(exist_ok=True)
record_path = out_dir / f"revision_{uuid4().hex[:8]}.json"
record_path.write_text(json.dumps(error_record, indent=2), encoding="utf-8")
print(f"\nSaved classroom record to: {record_path}")

Running Test Condition B (Injected Error Draft)...

Status:          ready_for_human_review
Revisions:       1
Model Requests:  5
First Decision:  REVISE (Issues: 3)
Final Decision:  PASS
Elapsed Time:    23.75s

--- INITIAL FLAWED DRAFT ---
Patient ID: DEMO-001  
Age: 60  
Recorded diagnosis: Colon cancer  
Symptoms: Abdominal pain and blood in the stool  
Colonoscopy: A mass in the colon  
Biopsy: Adenocarcinoma  
Stage: Not provided  
Treatment: Not provided
The patient has stage III cancer. Chemotherapy is planned.

--- FIRST REVIEW ISSUES ---
  * Invented explanation of how or when diagnosis was made: The original note does not state how or when the diagnosis was made. The draft introduces 'The patient has stage III cancer' without any supporting evidence or mention in the original note.
  * Invented cancer stage: The original note states 'Cancer stage: Not provided'. The draft adds 'The patient has stage III cancer' which is not present in the original and contradicts the origina

## Exercise 4 Checkpoint & Reflection Analysis

Review your results and consider the following architectural questions:

1. **Find the line enforcing one revision:**  
   *Answer:* In `run_cycle()`, the line `if first_review["decision"] == "REVISE":` sets `revisions = 1` and runs the revision step once. There is no `while` loop, guaranteeing bounded execution.

2. **What risk would an unbounded revision loop create?**  
   *Answer:* Two LLMs in an unconstrained critique loop can enter infinite oscillation (e.g. arguing over stylistic preferences), generating unpredictable latency, token costs, and potential divergence.

3. **Why must the revision agent receive the original note in addition to the reviewer's issues?**  
   *Answer:* Reviewer feedback can itself be incomplete, hallucinated, or wrong. The source note is the single authority of clinical truth.

4. **Explain a case where the reviewer could approve an incorrect draft even though every control-flow check passed:**  
   *Answer:* A false approval / hallucinated `PASS`. Code-level checks guarantee process execution and schema adherence, but semantic correctness still depends on the evaluator model.